# Notebook 01: Limpieza de Datos NHANES

Este notebook explora y limpia los 12 archivos `.xpt` (SAS) de la encuesta
NHANES 2017-2020. Aqui probamos la logica de los nodos antes de moverla
a los pipelines de Kedro.

**Flujo**:
1. Carga cruda (.xpt -> DataFrame)
2. Exploracion (nulos, ceros, tipos, distribuciones)
3. Seleccion de variables (segun `seleccion_variables_modelo.md`)
4. Limpieza (codigos NHANES missing, rangos, placeholder SAS)
5. Union de tablas por SEQN (master table)
6. Tratamiento de NaN (estructurales + imputacion + eliminacion)
7. Validacion final: 0 NaN

## 0. Setup

In [2]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

RAW_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "01_raw")
# Ajustar si se ejecuta desde otra ubicacion:
if not os.path.exists(RAW_DIR):
    RAW_DIR = r"d:\Programacion\Programación para la ciencia de datos\Ciencia-de-datos\Parcial3_2\Proyecto_Nhanes_Kedro\proyecto_nhanes_kedro\data\01_raw"

print(f"RAW_DIR: {RAW_DIR}")
print(f"Archivos: {sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.xpt')])}")

RAW_DIR: d:\Programacion\Programación para la ciencia de datos\Ciencia-de-datos\Parcial3_2\Proyecto_Nhanes_Kedro\proyecto_nhanes_kedro\data\01_raw
Archivos: ['P_ALQ.xpt', 'P_BIOPRO.xpt', 'P_BMX.xpt', 'P_BPXO.xpt', 'P_DEMO.xpt', 'P_DIQ.xpt', 'P_GHB.xpt', 'P_HDL.xpt', 'P_MCQ.xpt', 'P_PAQ.xpt', 'P_SLQ.xpt', 'P_SMQ.xpt']


## 1. Constantes NHANES

En los archivos `.xpt` de NHANES hay patrones especiales:
- Codigos `7`, `9`, `77`, `99`, `777`, `999`, `7777`, `9999` = "Se nego" / "No sabe"
- Un valor flotante `~5.397e-79` se usa como placeholder de NaN en archivos SAS
- Cada tabla tiene su propio diccionario de codificacion

In [3]:
# Valor placeholder de SAS XPT que actua como NaN
XPT_NAN = 5.397605346934028e-79

# Codigos NHANES que representan datos faltantes
NHANES_MISSING_CODES = {7, 9, 77, 99, 777, 999, 7777, 9999, 99999}

# Variables seleccionadas segun seleccion_variables_modelo.md
VARIABLES_SELECCIONADAS = {
    "P_DEMO": ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1", "DMDEDUC2", "INDFMPIR", "RIDSTATR"],
    "P_ALQ":  ["SEQN", "ALQ121", "ALQ130"],
    "P_BMX":  ["SEQN", "BMXBMI", "BMXWAIST"],
    "P_BPXO": ["SEQN", "BPXOSY1", "BPXODI1"],
    "P_DIQ":  ["SEQN", "DIQ010"],
    "P_GHB":  ["SEQN", "LBXGH"],
    "P_HDL":  ["SEQN", "LBDHDD"],
    "P_MCQ":  ["SEQN", "MCQ160A", "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220"],
    "P_PAQ":  ["SEQN", "PAQ650", "PAD680"],
    "P_SLQ":  ["SEQN", "SLD012", "SLQ030", "SLQ040"],
    "P_SMQ":  ["SEQN", "SMQ040", "SMD057"],
    "P_BIOPRO": ["SEQN", "LBXSCR", "LBXSBU", "LBXSATSI", "LBXSASSI", "LBXSAL"],
}

# Variables categoricas (donde los codigos 7/9/77/99 son "missing")
CATEGORICAS_CON_MISSING = {
    "P_DEMO": ["RIAGENDR", "RIDRETH1", "DMDEDUC2"],
    "P_ALQ":  ["ALQ121"],
    "P_DIQ":  ["DIQ010"],
    "P_MCQ":  ["MCQ160A", "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220"],
    "P_PAQ":  ["PAQ650"],
    "P_SLQ":  ["SLQ030", "SLQ040"],
    "P_SMQ":  ["SMQ040"],
}

print("Variables seleccionadas por tabla:")
for tabla, cols in VARIABLES_SELECCIONADAS.items():
    print(f"  {tabla}: {len(cols)-1} variables (+ SEQN)")

Variables seleccionadas por tabla:
  P_DEMO: 6 variables (+ SEQN)
  P_ALQ: 2 variables (+ SEQN)
  P_BMX: 2 variables (+ SEQN)
  P_BPXO: 2 variables (+ SEQN)
  P_DIQ: 1 variables (+ SEQN)
  P_GHB: 1 variables (+ SEQN)
  P_HDL: 1 variables (+ SEQN)
  P_MCQ: 7 variables (+ SEQN)
  P_PAQ: 2 variables (+ SEQN)
  P_SLQ: 3 variables (+ SEQN)
  P_SMQ: 2 variables (+ SEQN)
  P_BIOPRO: 5 variables (+ SEQN)


## 2. Carga y Exploracion Inicial

In [4]:
def cargar_xpt(nombre_archivo: str) -> pd.DataFrame:
    """Carga un archivo .xpt y retorna un DataFrame."""
    path = os.path.join(RAW_DIR, nombre_archivo)
    df = pd.read_sas(path, format="xport")
    return df


def perfil_rapido(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Genera un perfil rapido de un DataFrame."""
    info = []
    for col in df.columns:
        null_pct = df[col].isnull().mean() * 100
        zero_pct = (df[col] == 0).mean() * 100 if df[col].dtype in ["float64", "int64"] else 0
        xpt_nan_count = np.isclose(df[col], XPT_NAN, rtol=1e-10).sum() if df[col].dtype == "float64" else 0

        nhanes_missing = 0
        if df[col].dtype in ["float64", "int64"]:
            nhanes_missing = df[col].isin(NHANES_MISSING_CODES).sum()

        info.append({
            "tabla": nombre, "variable": col, "dtype": str(df[col].dtype),
            "no_nulos": df[col].notna().sum(), "nulos_pct": round(null_pct, 2),
            "ceros_pct": round(zero_pct, 2), "xpt_nan": xpt_nan_count,
            "nhanes_missing": nhanes_missing, "unique": df[col].nunique(),
            "min": df[col].min() if df[col].dtype in ["float64", "int64"] else None,
            "max": df[col].max() if df[col].dtype in ["float64", "int64"] else None,
        })
    return pd.DataFrame(info)

In [5]:
# Cargar todos los archivos raw
xpt_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".xpt")])
dfs_raw = {}
perfiles = []

print(f"{'='*70}")
print(f"CARGA DE ARCHIVOS CRUDOS")
print(f"{'='*70}\n")

for fname in xpt_files:
    nombre = fname.replace(".xpt", "")
    df = cargar_xpt(fname)
    dfs_raw[nombre] = df
    perfil = perfil_rapido(df, nombre)
    perfiles.append(perfil)
    print(f"  {nombre:12s}: {df.shape[0]:6,} filas x {df.shape[1]:2} columnas")

print(f"\n  TOTAL: {len(xpt_files)} archivos cargados")

CARGA DE ARCHIVOS CRUDOS

  P_ALQ       :  8,965 filas x 10 columnas
  P_BIOPRO    : 10,409 filas x 41 columnas
  P_BMX       : 14,300 filas x 22 columnas
  P_BPXO      : 11,656 filas x 12 columnas
  P_DEMO      : 15,560 filas x 29 columnas
  P_DIQ       : 14,986 filas x 28 columnas
  P_GHB       : 10,409 filas x  2 columnas
  P_HDL       : 12,198 filas x  3 columnas
  P_MCQ       : 14,986 filas x 63 columnas
  P_PAQ       :  9,693 filas x 17 columnas
  P_SLQ       : 10,195 filas x 11 columnas
  P_SMQ       : 11,137 filas x 16 columnas

  TOTAL: 12 archivos cargados


In [6]:
# Unir todos los perfiles y mostrar las variables del modelo
df_perfil = pd.concat(perfiles, ignore_index=True)

variables_usadas = set()
for cols in VARIABLES_SELECCIONADAS.values():
    variables_usadas.update(cols)

perfil_modelo = df_perfil[df_perfil["variable"].isin(variables_usadas)].copy()
print(f"\nPERFIL DE VARIABLES SELECCIONADAS PARA EL MODELO ({len(perfil_modelo)} variables):\n")
print(perfil_modelo.to_string(index=False))


PERFIL DE VARIABLES SELECCIONADAS PARA EL MODELO (46 variables):

   tabla variable   dtype  no_nulos  nulos_pct  ceros_pct  xpt_nan  nhanes_missing  unique       min       max
   P_ALQ     SEQN float64      8965       0.00       0.00        0               0    8965 109266.00 124822.00
   P_ALQ   ALQ121 float64      7503      16.31       0.00     1638            1369      13      0.00     99.00
   P_ALQ   ALQ130 float64      5863      34.60       0.00        0              67      16      1.00    999.00
P_BIOPRO     SEQN float64     10409       0.00       0.00        0               0   10409 109264.00 124822.00
P_BIOPRO LBXSATSI float64      9473       8.99       0.00        0             545     152      2.00    682.00
P_BIOPRO   LBXSAL float64      9477       8.95       0.00        0               0      32      2.10      5.40
P_BIOPRO LBXSASSI float64      9435       9.36       0.00        0              31     126      6.00    489.00
P_BIOPRO   LBXSBU float64      9473       8.9

## 3. Deteccion de Problemas

In [7]:
print("=" * 70)
print("DETECCION DE PROBLEMAS")
print("=" * 70)

# 3.1 Valores XPT NaN (~5.4e-79)
print("\n--- Valores XPT NaN (placeholder SAS ~5.4e-79) ---")
for nombre, df in dfs_raw.items():
    for col in df.select_dtypes(include=[np.number]).columns:
        count = np.isclose(df[col], XPT_NAN, rtol=1e-10).sum()
        if count > 0:
            print(f"  {nombre}.{col}: {count} valores XPT NaN ({count/len(df)*100:.1f}%)")

# 3.2 Codigos NHANES Missing en variables categoricas
print("\n--- Codigos NHANES Missing (7/9/77/99) en categoricas ---")
for tabla, cols in CATEGORICAS_CON_MISSING.items():
    if tabla in dfs_raw:
        df = dfs_raw[tabla]
        for col in cols:
            if col in df.columns:
                for code in sorted(NHANES_MISSING_CODES):
                    count = (df[col] == code).sum()
                    if count > 0:
                        print(f"  {tabla}.{col}: codigo {code} = {count} registros")

# 3.3 Variables con >50% nulos
print("\n--- Variables seleccionadas con >50% nulos ---")
for _, row in perfil_modelo.iterrows():
    if row["nulos_pct"] > 50:
        print(f"  {row['tabla']}.{row['variable']}: {row['nulos_pct']:.1f}% nulos")

DETECCION DE PROBLEMAS

--- Valores XPT NaN (placeholder SAS ~5.4e-79) ---
  P_ALQ.ALQ121: 1638 valores XPT NaN (18.3%)
  P_ALQ.ALQ142: 3381 valores XPT NaN (37.7%)
  P_ALQ.ALQ270: 1362 valores XPT NaN (15.2%)
  P_ALQ.ALQ280: 1538 valores XPT NaN (17.2%)
  P_ALQ.ALQ290: 502 valores XPT NaN (5.6%)
  P_ALQ.ALQ170: 4163 valores XPT NaN (46.4%)
  P_BIOPRO.LBDSATLC: 9471 valores XPT NaN (91.0%)
  P_BIOPRO.LBDSGTLC: 9472 valores XPT NaN (91.0%)
  P_BIOPRO.LBDSTBLC: 9474 valores XPT NaN (91.0%)
  P_DEMO.RIDAGEYR: 574 valores XPT NaN (3.7%)
  P_DEMO.RIDAGEMN: 57 valores XPT NaN (0.4%)
  P_DEMO.WTMECPRP: 1260 valores XPT NaN (8.1%)
  P_DEMO.INDFMPIR: 142 valores XPT NaN (0.9%)
  P_DIQ.DID250: 19 valores XPT NaN (0.1%)
  P_DIQ.DID260: 300 valores XPT NaN (2.0%)
  P_DIQ.DID341: 381 valores XPT NaN (2.5%)
  P_DIQ.DID350: 260 valores XPT NaN (1.7%)
  P_PAQ.PAD680: 3 valores XPT NaN (0.0%)
  P_SLQ.SLQ030: 2805 valores XPT NaN (27.5%)
  P_SLQ.SLQ040: 7424 valores XPT NaN (72.8%)
  P_SLQ.SLQ120: 1780 

## 4. Funciones de Limpieza por Tabla

Cada funcion toma un DataFrame crudo, selecciona las columnas
relevantes, reemplaza codigos NHANES, valida rangos y retorna
un DataFrame limpio.

In [8]:
def _base_clean(df: pd.DataFrame, nombre_tabla: str) -> pd.DataFrame:
    """Limpieza base: selecciona columnas, reemplaza XPT NaN y codigos NHANES."""
    # 1. Seleccionar columnas
    keep_cols = VARIABLES_SELECCIONADAS.get(nombre_tabla, [])
    available = [c for c in keep_cols if c in df.columns]
    result = df[available].copy()

    # 2. Reemplazar XPT NaN (~5.4e-79)
    for col in result.select_dtypes(include=[np.number]).columns:
        mask = np.isclose(result[col], XPT_NAN, rtol=1e-10)
        result.loc[mask, col] = np.nan

    # 3. Reemplazar codigos NHANES missing en categoricas
    cat_cols = CATEGORICAS_CON_MISSING.get(nombre_tabla, [])
    for col in cat_cols:
        if col in result.columns:
            result.loc[result[col].isin(NHANES_MISSING_CODES), col] = np.nan

    # 4. Asegurar SEQN como int sin nulos
    if "SEQN" in result.columns:
        result = result.dropna(subset=["SEQN"])
        result["SEQN"] = result["SEQN"].astype(int)

    return result.reset_index(drop=True)

In [9]:
def limpiar_demo(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_DEMO: filtra examinados, valida edad y genero."""
    result = _base_clean(df, "P_DEMO")
    if "RIDSTATR" in result.columns:
        result = result[result["RIDSTATR"] == 2].drop(columns=["RIDSTATR"])
    if "RIDAGEYR" in result.columns:
        result = result[(result["RIDAGEYR"] >= 0) & (result["RIDAGEYR"] <= 80)]
    if "RIAGENDR" in result.columns:
        result = result[result["RIAGENDR"].isin([1, 2]) | result["RIAGENDR"].isna()]
    return result.reset_index(drop=True)


def limpiar_alq(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_ALQ: frecuencia y cantidad de consumo."""
    result = _base_clean(df, "P_ALQ")
    if "ALQ121" in result.columns:
        result.loc[~result["ALQ121"].between(0, 10) & result["ALQ121"].notna(), "ALQ121"] = np.nan
    if "ALQ130" in result.columns:
        result.loc[~result["ALQ130"].between(1, 15) & result["ALQ130"].notna(), "ALQ130"] = np.nan
    return result.reset_index(drop=True)


def limpiar_bmx(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_BMX: valida rangos de IMC y cintura."""
    result = _base_clean(df, "P_BMX")
    if "BMXBMI" in result.columns:
        result.loc[~result["BMXBMI"].between(10, 100) & result["BMXBMI"].notna(), "BMXBMI"] = np.nan
    if "BMXWAIST" in result.columns:
        result.loc[~result["BMXWAIST"].between(30, 200) & result["BMXWAIST"].notna(), "BMXWAIST"] = np.nan
    return result.reset_index(drop=True)


def limpiar_bpxo(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_BPXO: valida rangos de presion."""
    result = _base_clean(df, "P_BPXO")
    if "BPXOSY1" in result.columns:
        result.loc[~result["BPXOSY1"].between(50, 250) & result["BPXOSY1"].notna(), "BPXOSY1"] = np.nan
    if "BPXODI1" in result.columns:
        result.loc[~result["BPXODI1"].between(0, 200) & result["BPXODI1"].notna(), "BPXODI1"] = np.nan
    return result.reset_index(drop=True)


def limpiar_diq(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_DIQ: diagnostico de diabetes."""
    result = _base_clean(df, "P_DIQ")
    if "DIQ010" in result.columns:
        result.loc[~result["DIQ010"].isin([1, 2, 3]) & result["DIQ010"].notna(), "DIQ010"] = np.nan
    return result.reset_index(drop=True)


def limpiar_ghb(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_GHB: HbA1c en porcentaje."""
    result = _base_clean(df, "P_GHB")
    if "LBXGH" in result.columns:
        result.loc[~result["LBXGH"].between(2.5, 20) & result["LBXGH"].notna(), "LBXGH"] = np.nan
    return result.reset_index(drop=True)


def limpiar_hdl(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_HDL: colesterol HDL."""
    result = _base_clean(df, "P_HDL")
    if "LBDHDD" in result.columns:
        result.loc[~result["LBDHDD"].between(5, 200) & result["LBDHDD"].notna(), "LBDHDD"] = np.nan
    return result.reset_index(drop=True)


def limpiar_mcq(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_MCQ: condiciones medicas previas. Convierte 1=Si, 2=No a 1/0."""
    result = _base_clean(df, "P_MCQ")
    binary_cols = ["MCQ160A", "MCQ160B", "MCQ160C", "MCQ160D",
                   "MCQ160E", "MCQ160F", "MCQ220"]
    for col in binary_cols:
        if col in result.columns:
            result[col] = result[col].map({1: 1, 2: 0})
    return result.reset_index(drop=True)


def limpiar_paq(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_PAQ: actividad fisica y sedentarismo."""
    result = _base_clean(df, "P_PAQ")
    if "PAQ650" in result.columns:
        result["PAQ650"] = result["PAQ650"].map({1: 1, 2: 0})
    if "PAD680" in result.columns:
        result.loc[~result["PAD680"].between(0, 1440) & result["PAD680"].notna(), "PAD680"] = np.nan
    return result.reset_index(drop=True)


def limpiar_slq(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_SLQ: horas de sueno, ronquidos y apnea."""
    result = _base_clean(df, "P_SLQ")
    if "SLD012" in result.columns:
        result.loc[~result["SLD012"].between(2, 14) & result["SLD012"].notna(), "SLD012"] = np.nan
    for col in ["SLQ030", "SLQ040"]:
        if col in result.columns:
            result.loc[~result[col].isin([0, 1, 2, 3]) & result[col].notna(), col] = np.nan
    return result.reset_index(drop=True)


def limpiar_smq(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_SMQ: estatus de fumador y cigarrillos/dia."""
    result = _base_clean(df, "P_SMQ")
    if "SMQ040" in result.columns:
        result.loc[~result["SMQ040"].isin([1, 2, 3]) & result["SMQ040"].notna(), "SMQ040"] = np.nan
    if "SMD057" in result.columns:
        result.loc[~result["SMD057"].between(1, 95) & result["SMD057"].notna(), "SMD057"] = np.nan
    return result.reset_index(drop=True)


def limpiar_biopro(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia P_BIOPRO: marcadores bioquimicos sericos."""
    result = _base_clean(df, "P_BIOPRO")
    rangos = {
        "LBXSCR":  (0.25, 14.97),  # Creatinina mg/dL
        "LBXSBU":  (2, 79),        # BUN mg/dL
        "LBXSATSI": (2, 682),      # ALT U/L
        "LBXSASSI": (6, 489),      # AST U/L
        "LBXSAL":  (2.1, 5.4),     # Albumina g/dL
    }
    for col, (lo, hi) in rangos.items():
        if col in result.columns:
            result.loc[~result[col].between(lo, hi) & result[col].notna(), col] = np.nan
    return result.reset_index(drop=True)


FUNCIONES_LIMPIEZA = {
    "P_DEMO": limpiar_demo, "P_ALQ": limpiar_alq, "P_BMX": limpiar_bmx,
    "P_BPXO": limpiar_bpxo, "P_DIQ": limpiar_diq, "P_GHB": limpiar_ghb,
    "P_HDL": limpiar_hdl, "P_MCQ": limpiar_mcq, "P_PAQ": limpiar_paq,
    "P_SLQ": limpiar_slq, "P_SMQ": limpiar_smq, "P_BIOPRO": limpiar_biopro,
}

## 5. Ejecutar Limpieza Individual

In [10]:
dfs_clean = {}

print(f"{'='*70}")
print(f"EJECUCION DE LIMPIEZA")
print(f"{'='*70}\n")
print(f"{'Tabla':<12} {'Raw':>8} {'Clean':>8} {'Cols':>6} {'Eliminados':>12}")
print(f"{'-'*50}")

for nombre, df_raw in dfs_raw.items():
    if nombre in FUNCIONES_LIMPIEZA:
        func = FUNCIONES_LIMPIEZA[nombre]
        df_clean = func(df_raw)
        dfs_clean[nombre] = df_clean
        eliminados = len(df_raw) - len(df_clean)
        print(f"{nombre:<12} {len(df_raw):>8,} {len(df_clean):>8,} {len(df_clean.columns):>6} {eliminados:>12,}")

EJECUCION DE LIMPIEZA

Tabla             Raw    Clean   Cols   Eliminados
--------------------------------------------------
P_ALQ           8,965    8,965      3            0
P_BIOPRO       10,409   10,409      6            0
P_BMX          14,300   14,300      3            0
P_BPXO         11,656   11,656      3            0
P_DEMO         15,560   13,772      6        1,788
P_DIQ          14,986   14,986      2            0
P_GHB          10,409   10,409      2            0
P_HDL          12,198   12,198      2            0
P_MCQ          14,986   14,986      8            0
P_PAQ           9,693    9,693      3            0
P_SLQ          10,195   10,195      4            0
P_SMQ          11,137   11,137      3            0


## 6. Crear Master Table (Union por SEQN)

Unimos todas las tablas limpias usando SEQN como clave.
P_DEMO es la tabla base (LEFT JOIN desde ella).

In [11]:
# Tabla base: DEMO (tiene todos los participantes examinados)
master = dfs_clean["P_DEMO"].copy()
print(f"Base: P_DEMO -> {len(master):,} filas x {master.shape[1]} cols")

# Unir las demas tablas
tablas_join = [k for k in dfs_clean if k != "P_DEMO"]
for nombre in tablas_join:
    df_tabla = dfs_clean[nombre]
    n_before = len(master)
    master = master.merge(df_tabla, on="SEQN", how="left")
    matched = master[df_tabla.columns[1]].notna().sum() if len(df_tabla.columns) > 1 else 0
    print(f"  + {nombre:12s}: {matched:>6,}/{n_before:,} coincidencias")

print(f"\nMaster table: {master.shape[0]:,} filas x {master.shape[1]} columnas")
print(f"Columnas: {list(master.columns)}")

Base: P_DEMO -> 13,772 filas x 6 cols
  + P_ALQ       :  4,496/13,772 coincidencias
  + P_BIOPRO    :  9,475/13,772 coincidencias
  + P_BMX       : 13,137/13,772 coincidencias
  + P_BPXO      : 10,352/13,772 coincidencias
  + P_DIQ       : 13,764/13,772 coincidencias
  + P_GHB       :  9,737/13,772 coincidencias
  + P_HDL       : 10,828/13,772 coincidencias
  + P_MCQ       :  8,521/13,772 coincidencias
  + P_PAQ       :  8,965/13,772 coincidencias
  + P_SLQ       :  9,364/13,772 coincidencias
  + P_SMQ       :  3,596/13,772 coincidencias

Master table: 13,772 filas x 34 columnas
Columnas: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'ALQ121', 'ALQ130', 'LBXSCR', 'LBXSBU', 'LBXSATSI', 'LBXSASSI', 'LBXSAL', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXODI1', 'DIQ010', 'LBXGH', 'LBDHDD', 'MCQ160A', 'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ220', 'PAQ650', 'PAD680', 'SLD012', 'SLQ030', 'SLQ040', 'SMQ040', 'SMD057']


In [12]:
# Reporte de nulos en master table ANTES del tratamiento
print(f"\n{'='*70}")
print(f"NULOS EN MASTER TABLE (antes del tratamiento)")
print(f"{'='*70}\n")

null_report = pd.DataFrame({
    "variable": master.columns,
    "nulos": master.isnull().sum().values,
    "pct_nulos": (master.isnull().mean() * 100).round(2).values,
}).sort_values("pct_nulos", ascending=False)

print(null_report[null_report["nulos"] > 0].to_string(index=False))
print(f"\nTotal celdas NaN: {master.isnull().sum().sum():,}")


NULOS EN MASTER TABLE (antes del tratamiento)

variable  nulos  pct_nulos
  SMD057  11762      85.41
  SLQ040  11669      84.73
  SMQ040  10176      73.89
  ALQ121   9276      67.35
  ALQ130   7919      57.50
  SLQ030   7547      54.80
 MCQ160D   5266      38.24
 MCQ160C   5254      38.15
 MCQ160A   5251      38.13
 MCQ160B   5249      38.11
 MCQ160F   5243      38.07
DMDEDUC2   5241      38.06
 MCQ160E   5241      38.06
  MCQ220   5232      37.99
  PAD680   4886      35.48
  PAQ650   4807      34.90
  SLD012   4408      32.01
LBXSASSI   4337      31.49
  LBXSBU   4299      31.22
LBXSATSI   4299      31.22
  LBXSCR   4297      31.20
  LBXSAL   4295      31.19
   LBXGH   4035      29.30
 BPXODI1   3420      24.83
 BPXOSY1   3420      24.83
  LBDHDD   2944      21.38
INDFMPIR   1917      13.92
BMXWAIST   1198       8.70
  BMXBMI    635       4.61
  DIQ010      8       0.06

Total celdas NaN: 153,531


## 7. Tratamiento de NaN (Estrategia Combinada)

### Paso 1: Rellenar NaN estructurales
Son NaN que tienen un significado logico (ej: no fuma -> cigarrillos = 0)

### Paso 2: Filtrar solo adultos (>= 18 anos)
Variables como DMDEDUC2 solo aplican a adultos 20+

### Paso 3: Imputar NaN restantes
- Continuas: mediana del grupo
- Categoricas: moda (valor mas frecuente)

### Paso 4: Eliminar filas con >50% NaN (si quedan)

In [13]:
# === PASO 1: Rellenar NaN Estructurales ===
print("PASO 1: Rellenando NaN estructurales...\n")
n_before = master.isnull().sum().sum()

# --- Tabaquismo (SMQ) ---
# Si SMQ040 es NaN, la persona NO fue preguntada -> nunca fumo 100+ cigarrillos
# Asignar SMQ040=3 ("No fuma en absoluto") y SMD057=0 (cigarrillos/dia)
mask_no_fumador = master["SMQ040"].isna()
master.loc[mask_no_fumador, "SMQ040"] = 3  # No fuma
master.loc[mask_no_fumador, "SMD057"] = 0  # 0 cigarrillos/dia
# Para los que si respondieron SMQ040 pero SMD057 es NaN (fuman "algunos dias")
mask_fuma_sin_cantidad = (master["SMQ040"].isin([1, 2])) & (master["SMD057"].isna())
master.loc[mask_fuma_sin_cantidad, "SMD057"] = 1  # Minimo: 1 cigarrillo/dia
print(f"  SMQ040: {mask_no_fumador.sum():,} no-fumadores rellenados")
print(f"  SMD057: {mask_fuma_sin_cantidad.sum():,} fumadores sin cantidad -> 1 cig/dia")

# --- Alcohol (ALQ) ---
# Si ALQ121 es NaN, la persona NO fue preguntada o no bebe
# Asignar ALQ121=0 ("Nunca en el ultimo ano") y ALQ130=0 (0 tragos/dia)
mask_no_bebe = master["ALQ121"].isna()
master.loc[mask_no_bebe, "ALQ121"] = 0  # Nunca bebe
master.loc[mask_no_bebe, "ALQ130"] = 0  # 0 tragos/dia
# Para los que beben pero no reportaron cantidad
mask_bebe_sin_cantidad = (master["ALQ121"] > 0) & (master["ALQ130"].isna())
master.loc[mask_bebe_sin_cantidad, "ALQ130"] = 1  # Minimo: 1 trago/dia
print(f"  ALQ121: {mask_no_bebe.sum():,} no-bebedores rellenados")
print(f"  ALQ130: {mask_bebe_sin_cantidad.sum():,} bebedores sin cantidad -> 1 trago/dia")

# --- Condiciones Medicas (MCQ) ---
# Si MCQ160A-F y MCQ220 son NaN, son personas jovenes (<40) sin historial
# Asignar 0 (No tiene la condicion)
mcq_cols = ["MCQ160A", "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F", "MCQ220"]
for col in mcq_cols:
    mask = master[col].isna()
    master.loc[mask, col] = 0  # No tiene la condicion
    if mask.sum() > 0:
        print(f"  {col}: {mask.sum():,} NaN -> 0 (sin condicion)")

# --- Sueno (SLQ) ---
# SLQ030 (ronquidos) y SLQ040 (apnea): NaN = no se pregunto o no tiene
master.loc[master["SLQ030"].isna(), "SLQ030"] = 0  # No ronca
master.loc[master["SLQ040"].isna(), "SLQ040"] = 0  # No tiene apnea
print(f"  SLQ030: ronquidos NaN -> 0 (no ronca)")
print(f"  SLQ040: apnea NaN -> 0 (sin apnea)")

# --- Diabetes (DIQ) ---
# DIQ010: muy pocos NaN (8), asignar 2 (No tiene diabetes)
mask_diq = master["DIQ010"].isna()
master.loc[mask_diq, "DIQ010"] = 2
print(f"  DIQ010: {mask_diq.sum():,} NaN -> 2 (no diabetico)")

n_after_step1 = master.isnull().sum().sum()
print(f"\n  NaN eliminados en Paso 1: {n_before - n_after_step1:,} ({(n_before - n_after_step1)/n_before*100:.1f}%)")
print(f"  NaN restantes: {n_after_step1:,}")

PASO 1: Rellenando NaN estructurales...

  SMQ040: 10,176 no-fumadores rellenados
  SMD057: 1,572 fumadores sin cantidad -> 1 cig/dia
  ALQ121: 9,276 no-bebedores rellenados
  ALQ130: 10 bebedores sin cantidad -> 1 trago/dia
  MCQ160A: 5,251 NaN -> 0 (sin condicion)
  MCQ160B: 5,249 NaN -> 0 (sin condicion)
  MCQ160C: 5,254 NaN -> 0 (sin condicion)
  MCQ160D: 5,266 NaN -> 0 (sin condicion)
  MCQ160E: 5,241 NaN -> 0 (sin condicion)
  MCQ160F: 5,243 NaN -> 0 (sin condicion)
  MCQ220: 5,232 NaN -> 0 (sin condicion)
  SLQ030: ronquidos NaN -> 0 (no ronca)
  SLQ040: apnea NaN -> 0 (sin apnea)
  DIQ010: 8 NaN -> 2 (no diabetico)

  NaN eliminados en Paso 1: 95,079 (61.9%)
  NaN restantes: 58,452


### Paso 2: Filtrar adultos (>= 18 anos)

Variables como DMDEDUC2 (nivel de educacion) solo aplican a adultos 20+.
Filtramos a participantes >= 18 anos para tener datos mas coherentes.

In [14]:
# === PASO 2: Filtrar adultos ===
print("PASO 2: Filtrando solo adultos (>= 18)...\n")
n_total = len(master)
master = master[master["RIDAGEYR"] >= 18].reset_index(drop=True)
print(f"  Antes: {n_total:,} filas")
print(f"  Despues: {len(master):,} filas (eliminados {n_total - len(master):,} menores)")

n_after_step2 = master.isnull().sum().sum()
print(f"  NaN restantes: {n_after_step2:,}")

PASO 2: Filtrando solo adultos (>= 18)...

  Antes: 13,772 filas
  Despues: 8,965 filas (eliminados 4,807 menores)
  NaN restantes: 9,255


In [15]:
# Reporte de nulos despues de filtrar adultos
print(f"\nNulos restantes por variable:")
null_remaining = master.isnull().sum()
null_remaining = null_remaining[null_remaining > 0].sort_values(ascending=False)
for col, n in null_remaining.items():
    pct = n / len(master) * 100
    print(f"  {col:15s}: {n:>5,} NaN ({pct:.1f}%)")


Nulos restantes por variable:
  INDFMPIR       : 1,329 NaN (14.8%)
  BPXOSY1        :   941 NaN (10.5%)
  BPXODI1        :   941 NaN (10.5%)
  LBXSASSI       :   744 NaN (8.3%)
  LBXSATSI       :   710 NaN (7.9%)
  LBXSBU         :   710 NaN (7.9%)
  LBXSCR         :   708 NaN (7.9%)
  LBXSAL         :   707 NaN (7.9%)
  LBDHDD         :   667 NaN (7.4%)
  BMXWAIST       :   516 NaN (5.8%)
  LBXGH          :   501 NaN (5.6%)
  DMDEDUC2       :   434 NaN (4.8%)
  BMXBMI         :   175 NaN (2.0%)
  PAD680         :    79 NaN (0.9%)
  SLD012         :    79 NaN (0.9%)
  SMD057         :    14 NaN (0.2%)


### Paso 3: Imputacion de NaN restantes

- **Variables continuas** (laboratorio, medidas): imputar con la **mediana**
- **Variables categoricas**: imputar con la **moda** (valor mas frecuente)

In [16]:
# === PASO 3: Imputacion ===
print("PASO 3: Imputando NaN restantes...\n")
n_before_impute = master.isnull().sum().sum()

# Clasificar variables
cols_continuas = ["RIDAGEYR", "INDFMPIR", "ALQ121", "ALQ130",
                  "BMXBMI", "BMXWAIST", "BPXOSY1", "BPXODI1",
                  "LBXGH", "LBDHDD", "PAD680", "SLD012", "SMD057",
                  "LBXSCR", "LBXSBU", "LBXSATSI", "LBXSASSI", "LBXSAL"]

cols_categoricas = ["RIAGENDR", "RIDRETH1", "DMDEDUC2", "DIQ010",
                    "MCQ160A", "MCQ160B", "MCQ160C", "MCQ160D",
                    "MCQ160E", "MCQ160F", "MCQ220",
                    "PAQ650", "SLQ030", "SLQ040", "SMQ040"]

# Imputar continuas con mediana
for col in cols_continuas:
    if col in master.columns:
        n_null = master[col].isnull().sum()
        if n_null > 0:
            mediana = master[col].median()
            master[col] = master[col].fillna(mediana)
            print(f"  {col:15s}: {n_null:>5,} NaN -> mediana = {mediana:.2f}")

# Imputar categoricas con moda
for col in cols_categoricas:
    if col in master.columns:
        n_null = master[col].isnull().sum()
        if n_null > 0:
            moda = master[col].mode()[0]
            master[col] = master[col].fillna(moda)
            print(f"  {col:15s}: {n_null:>5,} NaN -> moda = {moda:.0f}")

n_after_impute = master.isnull().sum().sum()
print(f"\n  NaN imputados en Paso 3: {n_before_impute - n_after_impute:,}")
print(f"  NaN restantes: {n_after_impute:,}")

PASO 3: Imputando NaN restantes...

  INDFMPIR       : 1,329 NaN -> mediana = 2.19
  BMXBMI         :   175 NaN -> mediana = 28.60
  BMXWAIST       :   516 NaN -> mediana = 99.00
  BPXOSY1        :   941 NaN -> mediana = 121.00
  BPXODI1        :   941 NaN -> mediana = 74.00
  LBXGH          :   501 NaN -> mediana = 5.60
  LBDHDD         :   667 NaN -> mediana = 51.00
  PAD680         :    79 NaN -> mediana = 300.00
  SLD012         :    79 NaN -> mediana = 7.50
  SMD057         :    14 NaN -> mediana = 0.00
  LBXSCR         :   708 NaN -> mediana = 0.84
  LBXSBU         :   710 NaN -> mediana = 14.00
  LBXSATSI       :   710 NaN -> mediana = 17.00
  LBXSASSI       :   744 NaN -> mediana = 19.00
  LBXSAL         :   707 NaN -> mediana = 4.10
  DMDEDUC2       :   434 NaN -> moda = 4

  NaN imputados en Paso 3: 9,255
  NaN restantes: 0


### Paso 4: Eliminar filas con NaN residuales (si quedan)

In [17]:
# === PASO 4: Eliminar residuales ===
print("PASO 4: Eliminando filas con NaN residuales...\n")
n_before_drop = len(master)
master = master.dropna().reset_index(drop=True)
n_dropped = n_before_drop - len(master)
print(f"  Filas eliminadas: {n_dropped:,}")
print(f"  Filas finales: {len(master):,}")

PASO 4: Eliminando filas con NaN residuales...

  Filas eliminadas: 0
  Filas finales: 8,965


## 8. Validacion Final: Cero NaN

In [18]:
print(f"\n{'='*70}")
print(f"VALIDACION FINAL")
print(f"{'='*70}\n")

total_nan = master.isnull().sum().sum()
print(f"  Total NaN en master table: {total_nan}")
print(f"  Shape final: {master.shape[0]:,} filas x {master.shape[1]} columnas")
print(f"  Columnas: {list(master.columns)}")

assert total_nan == 0, "ERROR: Aun quedan NaN en la tabla!"
print(f"\n  [VALIDADO] La master table tiene 0 NaN")


VALIDACION FINAL

  Total NaN en master table: 0
  Shape final: 8,965 filas x 34 columnas
  Columnas: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'ALQ121', 'ALQ130', 'LBXSCR', 'LBXSBU', 'LBXSATSI', 'LBXSASSI', 'LBXSAL', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXODI1', 'DIQ010', 'LBXGH', 'LBDHDD', 'MCQ160A', 'MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F', 'MCQ220', 'PAQ650', 'PAD680', 'SLD012', 'SLQ030', 'SLQ040', 'SMQ040', 'SMD057']

  [VALIDADO] La master table tiene 0 NaN


In [19]:
# Estadisticas descriptivas finales
print(f"\n{'='*70}")
print(f"ESTADISTICAS DESCRIPTIVAS FINALES")
print(f"{'='*70}\n")
print(master.describe().round(2).to_string())


ESTADISTICAS DESCRIPTIVAS FINALES

           SEQN  RIDAGEYR  RIAGENDR  RIDRETH1  DMDEDUC2  INDFMPIR  ALQ121  ALQ130  LBXSCR  LBXSBU  LBXSATSI  LBXSASSI  LBXSAL  BMXBMI  BMXWAIST  BPXOSY1  BPXODI1  DIQ010   LBXGH  LBDHDD  MCQ160A  MCQ160B  MCQ160C  MCQ160D  MCQ160E  MCQ160F  MCQ220  PAQ650  PAD680  SLD012  SLQ030  SLQ040  SMQ040  SMD057
count   8965.00   8965.00   8965.00   8965.00   8965.00   8965.00 8965.00 8965.00 8965.00 8965.00   8965.00   8965.00 8965.00 8965.00   8965.00  8965.00  8965.00 8965.00 8965.00 8965.00  8965.00  8965.00  8965.00  8965.00  8965.00  8965.00 8965.00 8965.00 8965.00 8965.00 8965.00 8965.00 8965.00 8965.00
mean  117107.85     49.47      1.51      3.27      3.57      2.52    2.86    1.34    0.90   14.83     21.67     21.61    4.06   29.86    100.32   124.05    74.73    1.88    5.82   53.27     0.29     0.04     0.04     0.02     0.04     0.05    0.10    0.25  333.90    7.61    1.38    0.39    2.69    3.58
std     4501.05     18.46      0.50      1.20      1

In [20]:
# Distribucion de variables categoricas
print(f"\n{'='*70}")
print(f"DISTRIBUCION DE VARIABLES CATEGORICAS")
print(f"{'='*70}")

for col in ["RIAGENDR", "RIDRETH1", "DMDEDUC2", "DIQ010", "PAQ650", "SMQ040"]:
    if col in master.columns:
        vc = master[col].value_counts().sort_index()
        print(f"\n  {col}:")
        for val, count in vc.items():
            pct = count / len(master) * 100
            print(f"    {val:.0f}: {count:>5,} ({pct:.1f}%)")


DISTRIBUCION DE VARIABLES CATEGORICAS

  RIAGENDR:
    1: 4,357 (48.6%)
    2: 4,608 (51.4%)

  RIDRETH1:
    1: 1,055 (11.8%)
    2:   934 (10.4%)
    3: 3,075 (34.3%)
    4: 2,378 (26.5%)
    5: 1,523 (17.0%)

  DMDEDUC2:
    1:   666 (7.4%)
    2:   947 (10.6%)
    3: 2,060 (23.0%)
    4: 3,205 (35.8%)
    5: 2,087 (23.3%)

  DIQ010:
    1: 1,327 (14.8%)
    2: 7,387 (82.4%)
    3:   251 (2.8%)

  PAQ650:
    0: 6,729 (75.1%)
    1: 2,236 (24.9%)

  SMQ040:
    1: 1,213 (13.5%)
    2:   359 (4.0%)
    3: 7,393 (82.5%)


In [21]:
# Vista previa de las primeras 10 filas
print(f"\n{'='*70}")
print(f"VISTA PREVIA (primeras 10 filas)")
print(f"{'='*70}\n")
print(master.head(10).to_string(index=False))


VISTA PREVIA (primeras 10 filas)

  SEQN  RIDAGEYR  RIAGENDR  RIDRETH1  DMDEDUC2  INDFMPIR  ALQ121  ALQ130  LBXSCR  LBXSBU  LBXSATSI  LBXSASSI  LBXSAL  BMXBMI  BMXWAIST  BPXOSY1  BPXODI1  DIQ010  LBXGH  LBDHDD  MCQ160A  MCQ160B  MCQ160C  MCQ160D  MCQ160E  MCQ160F  MCQ220  PAQ650  PAD680  SLD012  SLQ030  SLQ040  SMQ040  SMD057
109266     29.00      2.00      5.00      5.00      5.00   10.00    1.00    0.63    8.00     15.00     14.00    3.80   37.80    117.90    99.00    56.00    2.00   5.20   56.00     0.00     0.00     0.00     0.00     0.00     0.00    0.00    1.00  480.00    7.50    1.00    0.00    3.00    0.00
109271     49.00      1.00      3.00      2.00      2.19    0.00    0.00    0.78    8.00      8.00     14.00    3.80   29.70    120.40   102.00    65.00    2.00   5.60   33.00     1.00     0.00     0.00     0.00     0.00     0.00    0.00    0.00   60.00   10.00    0.00    0.00    1.00    1.00
109273     36.00      1.00      3.00      4.00      0.83    0.00    0.00    0.92   

## 9. Resumen del Proceso

### Limpieza aplicada:
- **Codigos NHANES missing** (7, 9, 77, 99) -> reemplazados por NaN en categoricas
- **Placeholder SAS** (~5.4e-79) -> reemplazado por NaN
- **Rangos medicos invalidos** -> reemplazados por NaN
- **Respuestas binarias** (Si=1/No=2) -> convertidas a (1/0)
- **Filtro DEMO** -> solo participantes examinados (RIDSTATR=2)

### Tratamiento de NaN:
- **Estructurales** -> rellenados con valores logicos (no fuma->0, no bebe->0)
- **Filtro adultos** -> solo >= 18 anos
- **Imputacion** -> mediana (continuas) / moda (categoricas)
- **Residuales** -> eliminados

### Resultado final:
- 0 valores NaN
- Datos listos para el pipeline de Kedro

In [22]:
print(f"\n[OK] Notebook de limpieza completado exitosamente.")
print(f"   Tablas procesadas: {len(dfs_clean)}")
print(f"   Master table final: {master.shape[0]:,} filas x {master.shape[1]} columnas")
print(f"   Total NaN: {master.isnull().sum().sum()}")


[OK] Notebook de limpieza completado exitosamente.
   Tablas procesadas: 12
   Master table final: 8,965 filas x 34 columnas
   Total NaN: 0
